# Treated MonoCulture Math Modelling — 20k IC50 (Julia)

Uses DifferentialEquations.jl + Optimization.jl with an ODEProblem and a Hill-type drug effect, mirroring the style of the untreated notebook.

In [3]:
using Pkg
# Toggle installs if you hit missing packages
if false
    Pkg.activate(temp=true)
    Pkg.add([
        "CSV", "DataFrames", "Statistics", "DifferentialEquations",
        "Optimization", "OptimizationOptimJL", "BlackBoxOptim", "Plots"
    ])
end
using CSV, DataFrames, Statistics
using DifferentialEquations, Optimization, OptimizationOptimJL, BlackBoxOptim
using Plots
default(fmt=:png, legend=:topright, lw=2, size=(900,550))

ROOT = raw"c:/Users/MainFrameTower/Desktop/CancerGrowthDynamics"
UNT_PARAMS = joinpath(ROOT, "Modelling Data Notebooks", "Untreated MonoCulture", "untreated_logistic_params_20k.csv")
TREATED_DIR = joinpath(ROOT, "Processed_Datasets", "Treated MonoCulture", "20k", "IC50", "Averages")
OUT_DIR = joinpath(ROOT, "Modelling Data Notebooks", "Treated MonoCulture", "20k", "IC50")
mkpath(OUT_DIR)

function load_day_averages(path::AbstractString)
    df = CSV.read(path, DataFrame)
    daycol = :Day ∈ names(df) ? :Day : Symbol(first(filter(n->occursin("day", lowercase(String(n))), names(df))))
    valcol = Symbol("Mean Cells") ∈ names(df) ? Symbol("Mean Cells") : Symbol(first(filter(n->occursin("mean", lowercase(String(n))) && occursin("cells", lowercase(String(n))), names(df))))
    x = Float64.(df[!, daycol]); y = Float64.(df[!, valcol])
    perm = sortperm(x); x=x[perm]; y=y[perm]
    return x, y
end

function load_untreated_rK()
    df = CSV.read(UNT_PARAMS, DataFrame)
    return Dict(Symbol(r.cell_line)=> (r.r, r.K) for r in eachrow(df))
end

# Logistic growth with Hill-type drug effect as reduction in effective growth rate
# du/dt = (r * (1 - H(dose; IC50, n)))*u*(1 - u/K)
hill(dose, IC50, n) = 1.0 / (1.0 + (IC50 / max(dose, eps()))^n)
function treated_logistic!(du, u, p, t)
    r, K, IC50, n, dose = p
    eff = r * (1 - hill(dose, IC50, n))
    du[1] = eff * u[1] * (1 - u[1]/K)
end

# Build and solve with given parameters at sample times x
function solve_model(x::Vector{Float64}, y::Vector{Float64}, r, K; IC50=1.0, n=1.0, dose=1.0)
    u0 = [max(y[1], eps())]; tspan=(x[1], x[end])
    p = [r, K, IC50, n, dose]
    prob = ODEProblem(treated_logistic!, u0, tspan, p)
    sol = solve(prob, Rosenbrock23(); saveat=x, reltol=1e-9, abstol=1e-9)
    return sol
end

# Objective for Optimization.jl (squared error)
function objective(θ, x, y, r, K, dose)
    IC50, n = θ
    sol = solve_model(x, y, r, K; IC50=IC50, n=n, dose=dose)
    pred = getindex.(sol.u, 1)
    return sum(abs2, y .- pred)
end

# Helper to fit IC50 and n for a given cell line and CSV
function fit_ic50_for_csv(csv_path::AbstractString, rK::Tuple{Float64,Float64}; dose=1.0)
    x, y = load_day_averages(csv_path)
    r, K = rK
    # Initial guesses bounded: IC50 in [1e-3, 1e3], n in [0.1, 5]
    θ0 = [1.0, 1.0]
    lower = [1e-3, 0.1]; upper=[1e3, 5.0]
    loss(θ) = objective(θ, x, y, r, K, dose)
    optf = OptimizationFunction((θ, p)->loss(θ), Optimization.AutoForwardDiff())
    prob = Optimization.OptimizationProblem(optf, θ0)
    res = Optimization.solve(prob, NelderMead(), lower, upper; maxiters=2000)
    θ̂ = res.u
    IC50̂ = θ̂[1]
    n̂ = θ̂[2]
    sol̂ = solve_model(x, y, r, K; IC50=IC50̂, n=n̂, dose=dose)
    ssr = objective(θ̂, x, y, r, K, dose)
    return (; x, y, r, K, IC50=IC50̂, n=n̂, ssr, sol=sol̂)
end

# Wire up actual 20k/IC50 treated averages CSVs
naive_csv = joinpath(TREATED_DIR, "A2780Naive_day_averages.csv")
cis_csv   = joinpath(TREATED_DIR, "A2780cis_day_averages.csv")

# If filenames differ, you can update these two variables to match the actual files.
println("Looking for:\n  " * naive_csv * "\n  " * cis_csv)

rk = load_untreated_rK()
@assert haskey(rk, :A2780Naive) && haskey(rk, :A2780cis) "Missing untreated r,K export for 20k"

fits = Dict{Symbol,Any}()
if isfile(naive_csv)
    fits[:A2780Naive] = fit_ic50_for_csv(naive_csv, rk[:A2780Naive]; dose=1.0)
else
    @warn "Naive treated 20k IC50 CSV not found" naive_csv
end
if isfile(cis_csv)
    fits[:A2780cis] = fit_ic50_for_csv(cis_csv, rk[:A2780cis]; dose=1.0)
else
    @warn "Cis treated 20k IC50 CSV not found" cis_csv
end

# Plot
for (k, f) in fits
    plt = scatter(f.x, f.y; label=String(k)*" data", xlabel="Day", ylabel="Cells", title=String(k)*" — 20k IC50 fit")
    plot!(plt, f.sol.t, getindex.(f.sol.u,1); label="Model")
    display(plt)
end

# Save summary CSV
out_csv = joinpath(OUT_DIR, "treated_ic50_20k_fit_params.csv")
rows = DataFrame(cell_line=String[], r=Float64[], K=Float64[], IC50=Float64[], n=Float64[], SSR=Float64[])
for name in (:A2780Naive, :A2780cis)
    if haskey(fits, name)
        f = fits[name]
        push!(rows, (String(name), f.r, f.K, f.IC50, f.n, f.ssr))
    end
end
CSV.write(out_csv, rows)
println("Saved params → " * out_csv)

Looking for:
  c:/Users/MainFrameTower/Desktop/CancerGrowthDynamics\Processed_Datasets\Treated MonoCulture\20k\IC50\Averages\A2780Naive_day_averages.csv
  c:/Users/MainFrameTower/Desktop/CancerGrowthDynamics\Processed_Datasets\Treated MonoCulture\20k\IC50\Averages\A2780cis_day_averages.csv
NelderMead{Optim.AffineSimplexer, Optim.AdaptiveParameters}(

LoadError: Optimization algorithm not found. Either the chosen algorithm is not a valid solver
choice for the `OptimizationProblem`, or the Optimization solver library is not loaded.
Make sure that you have loaded an appropriate Optimization.jl solver library, for example,
`solve(prob,Optim.BFGS())` requires `using OptimizationOptimJL` and
`solve(prob,Adam())` requires `using OptimizationOptimisers`.

For more information, see the Optimization.jl documentation: https://docs.sciml.ai/Optimization/stable/.

Chosen Optimizer: 

Optim.AffineSimplexer(0.025, 0.5), Optim.AdaptiveParameters(1.0, 1.0, 0.75, 1.0))

In [ ]:
# Quick check: use discovered filenames
TREATED_DIR = joinpath(ROOT, "Processed_Datasets", "Treated MonoCulture", "20k", "IC50", "Averages")
naive_csv = joinpath(TREATED_DIR, "A2780Naive_day_averages.csv")
cis_csv   = joinpath(TREATED_DIR, "A2780cis_day_averages.csv")
println("Using:\n  " * naive_csv * "\n  " * cis_csv)